## Prerequisites

**Run these notebooks first:**
1. ✅ `preprocessing_FIXED.ipynb` (creates preprocessed_canonical/)
2. ✅ `01_setup_and_config.ipynb` (sets up environment)

If you haven't run those, **stop here and run them first!**

In [ ]:
# Restore variables from setup notebook
%store -r config
%store -r device

print("✓ Config and device loaded from setup notebook")

## 1. Import Everything

In [ ]:
import sys
import os
sys.path.insert(0, os.getcwd())

from utils import (
    set_seed, get_dataloaders, validate_checkpoint_fresh,
    train_one_epoch, validate, evaluate_model,
    plot_confusion_matrix, plot_training_curves, plot_roc_curves
)
from models import TumorNetLite, print_model_summary

import torch
import torch.nn as nn
import torch.optim as optim
from datetime import datetime
import json
import time
from tqdm.notebook import tqdm

print("✓ All modules imported successfully")

## 2. Set Random Seed (Again for Safety)

In [ ]:
set_seed(config['reproducibility']['seed'], config['reproducibility']['deterministic'])

## 3. Load Data

**CRITICAL:** Loads ONLY from `preprocessed_canonical/` directory

In [ ]:
# Load dataloaders
train_loader, val_loader, internal_test_loader, heldout_test_loader = get_dataloaders(
    config=config,
    preprocessed_dir=config['paths']['preprocessed_data']
)

class_names = config['data']['class_names']
num_classes = len(class_names)

print(f"\n✓ Data loaded successfully!")
print(f"Classes: {class_names}")

## 4. Visualize Sample Data

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Get a batch
images, labels = next(iter(train_loader))

# Denormalize for visualization
mean = torch.tensor(config['data']['normalization']['mean']).view(3, 1, 1)
std = torch.tensor(config['data']['normalization']['std']).view(3, 1, 1)
images_denorm = images * std + mean

# Plot first 8 images
fig, axes = plt.subplots(2, 4, figsize=(15, 8))
for idx, ax in enumerate(axes.flat):
    if idx < len(images):
        img = images_denorm[idx].permute(1, 2, 0).numpy()
        img = np.clip(img, 0, 1)
        ax.imshow(img)
        ax.set_title(f"{class_names[labels[idx]]}")
        ax.axis('off')
plt.tight_layout()
plt.show()

print(f"Image shape: {images.shape}")
print(f"Batch size: {len(images)}")

## 5. Create Model

**CRITICAL:** Validates checkpoint doesn't exist (ensures fresh training)

In [ ]:
# Create unique experiment name with timestamp
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
experiment_name = f"tumornet_lite_main_{timestamp}"
checkpoint_path = os.path.join(config['paths']['checkpoints'], f"{experiment_name}.pth")

# Validate checkpoint doesn't exist
validate_checkpoint_fresh(checkpoint_path, force_fresh=True)

print(f"\nExperiment: {experiment_name}")
print(f"Checkpoint: {checkpoint_path}")

In [ ]:
# Create TumorNet-Lite model
model = TumorNetLite(
    num_classes=num_classes,
    pretrained=False,  # Start from scratch
    in_channels=3,
    base_channels=64
)

model = model.to(device)
print_model_summary(model, "TumorNet-Lite")

## 6. Training Setup

In [ ]:
# Loss function
criterion = nn.CrossEntropyLoss()

# Optimizer
optimizer = optim.AdamW(
    model.parameters(),
    lr=config['optimizer']['learning_rate'],
    weight_decay=config['optimizer']['weight_decay'],
    betas=tuple(config['optimizer']['betas'])
)

# Scheduler
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode=config['scheduler']['mode'],
    factor=config['scheduler']['factor'],
    patience=config['scheduler']['patience'],
    verbose=True
)

# Mixed precision scaler
scaler = torch.cuda.amp.GradScaler() if config['training']['mixed_precision'] else None

# Training parameters
max_epochs = config['training']['max_epochs']
early_stopping_patience = config['training']['early_stopping']['patience']
max_grad_norm = config['training']['gradient_clipping']['max_norm']

print("✓ Training setup complete")
print(f"  Max epochs: {max_epochs}")
print(f"  Early stopping patience: {early_stopping_patience}")
print(f"  Initial LR: {config['optimizer']['learning_rate']}")
print(f"  Mixed precision: {config['training']['mixed_precision']}")

## 7. Training Loop

**Protocol:** Train on `train/`, validate on `val/`, never touch `heldout_test/` until final evaluation

In [ ]:
# Training history
history = {
    'train_loss': [],
    'val_loss': [],
    'train_acc': [],
    'val_acc': [],
    'learning_rates': []
}

best_val_acc = 0.0
best_epoch = 0
patience_counter = 0

print("="*80)
print("STARTING TRAINING")
print("="*80)
print(f"Device: {device}")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")
print("="*80)

In [ ]:
# Main training loop
for epoch in range(1, max_epochs + 1):
    epoch_start = time.time()
    
    # Train
    train_loss, train_acc = train_one_epoch(
        model, train_loader, criterion, optimizer,
        device, scaler, max_grad_norm, epoch
    )
    
    # Validate
    val_loss, val_acc = validate(
        model, val_loader, criterion, device
    )
    
    # Update scheduler
    scheduler.step(val_loss)
    current_lr = optimizer.param_groups[0]['lr']
    
    # Save history
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)
    history['learning_rates'].append(current_lr)
    
    # Print progress
    epoch_time = time.time() - epoch_start
    print(f"\nEpoch [{epoch}/{max_epochs}] - {epoch_time:.1f}s")
    print(f"  Train: Loss={train_loss:.4f} | Acc={train_acc:.2f}%")
    print(f"  Val:   Loss={val_loss:.4f} | Acc={val_acc:.2f}%")
    print(f"  LR: {current_lr:.6f}")
    
    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_epoch = epoch
        patience_counter = 0
        
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'best_val_acc': best_val_acc,
            'history': history
        }
        torch.save(checkpoint, checkpoint_path)
        print(f"  ✓ Best model saved! (Val Acc: {val_acc:.2f}%)")
    else:
        patience_counter += 1
        print(f"  Patience: {patience_counter}/{early_stopping_patience}")
    
    # Early stopping
    if patience_counter >= early_stopping_patience:
        print(f"\n⚠️  Early stopping at epoch {epoch}")
        break
    
    print("-" * 80)

print("\n" + "="*80)
print("TRAINING COMPLETE")
print("="*80)
print(f"Best val accuracy: {best_val_acc:.2f}% (epoch {best_epoch})")

## 8. Visualize Training

In [ ]:
# Plot training curves
results_dir = config['paths']['results']
curves_path = os.path.join(results_dir, f"{experiment_name}_training_curves.png")
plot_training_curves(history, save_path=curves_path)

In [ ]:
# Plot learning rate schedule
plt.figure(figsize=(10, 5))
plt.plot(range(1, len(history['learning_rates']) + 1), history['learning_rates'], 'b-o')
plt.xlabel('Epoch')
plt.ylabel('Learning Rate')
plt.title('Learning Rate Schedule')
plt.yscale('log')
plt.grid(True, alpha=0.3)
plt.show()

## 9. Load Best Model

In [ ]:
checkpoint = torch.load(checkpoint_path)
model.load_state_dict(checkpoint['model_state_dict'])
print(f"✓ Loaded best model from epoch {checkpoint['epoch']}")
print(f"  Best validation accuracy: {checkpoint['best_val_acc']:.2f}%")

## 10. Evaluate on Internal Test Set

In [ ]:
from utils import print_experiment_summary

print("Evaluating on internal test set...")
internal_results = evaluate_model(
    model, internal_test_loader, device, class_names
)

print_experiment_summary(
    "TumorNet-Lite (Internal Test)",
    internal_results,
    class_names
)

In [ ]:
# Confusion matrix
cm_path = os.path.join(results_dir, f"{experiment_name}_cm_internal.png")
plot_confusion_matrix(
    internal_results['confusion_matrix'],
    class_names,
    save_path=cm_path,
    title='Confusion Matrix - Internal Test'
)

In [ ]:
# ROC curves
roc_path = os.path.join(results_dir, f"{experiment_name}_roc_internal.png")
roc_auc = plot_roc_curves(
    internal_results['labels'],
    internal_results['probabilities'],
    class_names,
    save_path=roc_path
)

## 11. FINAL EVALUATION - Held-Out Test Set

**⚠️ CRITICAL: This is the ONLY time we touch the held-out test set!**

This provides the unbiased performance estimate for publication.

In [ ]:
print("="*80)
print("FINAL EVALUATION ON HELD-OUT TEST SET")
print("="*80)
print("⚠️  First and only time evaluating this test set!")
print("="*80 + "\n")

heldout_results = evaluate_model(
    model, heldout_test_loader, device, class_names
)

print_experiment_summary(
    "TumorNet-Lite (Held-Out Test - FINAL)",
    heldout_results,
    class_names
)

In [ ]:
# Confusion matrix for held-out test
cm_path_heldout = os.path.join(results_dir, f"{experiment_name}_cm_heldout_FINAL.png")
plot_confusion_matrix(
    heldout_results['confusion_matrix'],
    class_names,
    save_path=cm_path_heldout,
    title='Confusion Matrix - Held-Out Test (FINAL)'
)

In [ ]:
# ROC curves for held-out test
roc_path_heldout = os.path.join(results_dir, f"{experiment_name}_roc_heldout_FINAL.png")
roc_auc_heldout = plot_roc_curves(
    heldout_results['labels'],
    heldout_results['probabilities'],
    class_names,
    save_path=roc_path_heldout
)

## 12. Save Complete Results

In [ ]:
import numpy as np

# Compile all results
complete_results = {
    'experiment_name': experiment_name,
    'timestamp': timestamp,
    'model': 'TumorNet-Lite',
    'total_parameters': sum(p.numel() for p in model.parameters()),
    'training': {
        'total_epochs': len(history['train_loss']),
        'best_epoch': best_epoch,
        'best_val_acc': float(best_val_acc),
        'final_train_acc': float(history['train_acc'][-1]),
        'final_val_acc': float(history['val_acc'][-1])
    },
    'internal_test': {
        'accuracy': float(internal_results['accuracy']),
        'classification_report': internal_results['classification_report'],
        'mean_auc': float(np.mean(list(roc_auc.values())))
    },
    'heldout_test_FINAL': {
        'accuracy': float(heldout_results['accuracy']),
        'classification_report': heldout_results['classification_report'],
        'mean_auc': float(np.mean(list(roc_auc_heldout.values())))
    }
}

# Save to JSON
results_path = os.path.join(results_dir, f"{experiment_name}_complete_results.json")
with open(results_path, 'w') as f:
    json.dump(complete_results, f, indent=2)

print(f"✓ Results saved to {results_path}")

## 13. Final Summary

In [ ]:
print("\n" + "="*80)
print("EXPERIMENT COMPLETE")
print("="*80)
print(f"\nExperiment: {experiment_name}")
print(f"Model: TumorNet-Lite")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"\nTraining:")
print(f"  Epochs: {len(history['train_loss'])}")
print(f"  Best epoch: {best_epoch}")
print(f"  Best val acc: {best_val_acc:.2f}%")
print(f"\nInternal Test:")
print(f"  Accuracy: {internal_results['accuracy']:.2f}%")
print(f"  Mean AUC: {np.mean(list(roc_auc.values())):.4f}")
print(f"\nHeld-Out Test (FINAL):")
print(f"  Accuracy: {heldout_results['accuracy']:.2f}%")
print(f"  Mean AUC: {np.mean(list(roc_auc_heldout.values())):.4f}")
print("\n" + "="*80)
print("✓ All bugs fixed!")
print("✓ Reproducible results!")
print("✓ No data leakage!")
print("✓ Fresh training confirmed!")
print("="*80)